In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
import numpy as np

# Define a simple CNN model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Define the Mean Teacher model
class MeanTeacher(nn.Module):
    def __init__(self, student_model, teacher_model, alpha=0.99):
        super(MeanTeacher, self).__init__()
        self.student = student_model
        self.teacher = teacher_model
        self.alpha = alpha

    def update_teacher(self):
        # Exponential moving average (EMA) update for the teacher model
        for student_param, teacher_param in zip(self.student.parameters(), self.teacher.parameters()):
            teacher_param.data.mul_(self.alpha).add_(student_param.data, alpha=1 - self.alpha)

    def forward(self, x):
        return self.student(x)

# Hyperparameters
batch_size = 64
learning_rate = 0.01
num_epochs = 20
alpha = 0.99  # EMA decay rate
labeled_ratio = 0.1  # Percentage of labeled data

# Data Augmentation
transform_labeled = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_unlabeled = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_labeled)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_labeled)

# Split into labeled and unlabeled data
num_labeled = int(len(train_dataset) * labeled_ratio)
indices = np.arange(len(train_dataset))
np.random.shuffle(indices)
labeled_indices = indices[:num_labeled]
unlabeled_indices = indices[num_labeled:]

labeled_dataset = Subset(train_dataset, labeled_indices)
unlabeled_dataset = Subset(train_dataset, unlabeled_indices)

# Create DataLoaders
labeled_loader = DataLoader(labeled_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Initialize models
student_model = SimpleCNN().cuda()
teacher_model = SimpleCNN().cuda()
mean_teacher = MeanTeacher(student_model, teacher_model, alpha).cuda()

# Loss functions
criterion_supervised = nn.CrossEntropyLoss()
criterion_consistency = nn.MSELoss()

# Optimizer
optimizer = optim.SGD(mean_teacher.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)

# Training loop
for epoch in range(num_epochs):
    mean_teacher.train()
    total_loss = 0
    for (labeled_data, labeled_targets), (unlabeled_data, _) in zip(labeled_loader, unlabeled_loader):
        labeled_data, labeled_targets = labeled_data.cuda(), labeled_targets.cuda()
        unlabeled_data = unlabeled_data.cuda()

        # Forward pass for labeled data
        labeled_outputs = mean_teacher(labeled_data)
        supervised_loss = criterion_supervised(labeled_outputs, labeled_targets)

        # Forward pass for unlabeled data (with consistency regularization)
        with torch.no_grad():
            teacher_outputs = mean_teacher.teacher(unlabeled_data)
        student_outputs = mean_teacher(unlabeled_data)
        consistency_loss = criterion_consistency(student_outputs, teacher_outputs)

        # Total loss
        loss = supervised_loss + consistency_loss

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update teacher model
        mean_teacher.update_teacher()

        total_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(labeled_loader):.4f}')

print("Training complete!")

